In [1]:
import os
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt

In [2]:
def get_table(spreadsheet_id, sheet_name, file_path):

    url = f"https://docs.google.com/spreadsheets/d/{spreadsheet_id}/export?format=xlsx"

    parts = file_path.split('/')
    
    path, file = '/'.join(parts[:-1]), parts[-1]
    
    if file in os.listdir(path):
        df = pd.read_csv(file_path)
    else:
        df = pd.read_excel(url, sheet_name=sheet_name)
        df.to_csv(file_path, index=False)

    return df

# 1. Кейс. По ссылке находится Массив по продажам. Проанализируйте его, выявите проблемы, обозначьте выводы.

In [3]:
SPREADSHEET_ID = "1rWBMGgYoex3yb3OFDiWD02kWcwIjahtn"
SHEET_NAME = "TDSheet"
FILE_PATH = "./МАССИВ_СТРУКТУРА ПРОДАЖ_отпр.csv"

df = get_table(SPREADSHEET_ID, SHEET_NAME, FILE_PATH)

df["Контрольная Сумма с НДС"] = round(df["Вес"] * df["Цена за кг СНДС"], 2)
df["Абсолютное расхождение"] = abs(df["Сумма с НДС"] - df["Контрольная Сумма с НДС"])
df["Относительное расхождение"] = df["Абсолютное расхождение"] / df["Контрольная Сумма с НДС"]

discrepancy = df["Относительное расхождение"].describe()

print("Среднее расхождение =", discrepancy["mean"].round(4)*100)
print("Стандартное отклонение в расхождениях =", discrepancy["std"].round(4) * 100)
print("Максимально расхождение =", discrepancy["max"].round(4) * 100)

result = df[df["Относительное расхождение"] > discrepancy["mean"] + 2*discrepancy["std"]].sort_values("Относительное расхождение", ascending=False)
print("Всего расхождений =", result.shape[0])

Среднее расхождение = 1.78
Стандартное отклонение в расхождениях = 3.49
Максимально расхождение = 30.0
Всего расхождений = 18


In [4]:
groups = [
    "Месяц", "Канал сбыта", "Производитель (из тов. категории)", "Номенклатура нормализованная", "Публичное наименование нормализованное", 
    "Сегмент.Код", "Менеджер", "Тип упаковки", "Марка (Бренд)", "Форма продукта"
]
for gr in groups:
    diff = pd.concat([df.value_counts(gr), result.value_counts(gr)], axis=1)
    diff.columns = ["Источник", "Расхождения"]
    diff = diff[~diff["Расхождения"].isna()]
    
    print(diff, end='\n\n')

       Источник  Расхождения
Месяц                       
10          211          4.0
2           187         14.0

                    Источник  Расхождения
Канал сбыта                              
1.1 Дистрибьюторы        369         16.0
1.2.2 МСК партнеры       109          1.0
1.3.2 Крупный Опт         21          1.0

                                   Источник  Расхождения
Производитель (из тов. категории)                       
Кобрин                                  553           18

                                                 Источник  Расхождения
Номенклатура нормализованная                                          
ГРОССМЕЙСТЕР 50 МВ НИЗКИЙ ЦИЛИНДР КОЛЕСО КОБРИН        28          4.0
МААСДАМ 45 МВ НИЗКИЙ ЦИЛИНДР КОЛЕСО КОБРИН             24          7.0
МОНДОР GRANO 50 МВ НИЗКИЙ ЦИЛИНДР КОЛЕСО КОБРИН        13          7.0

                                        Источник  Расхождения
Публичное наименование нормализованное                       
ГРОССМЕЙСТЕР МВ КОЛЕ

## Проблема:
- Есть 18 записей у которых расхожденние в "Сумме с НДС" превышает 10%, что выходит за рамки нормальной погрешности в данных
- У 5 записей расхождение доходит до 30%
- Большинство расхождений во 2м месяце (14), у дистрибъютора "1.1" (16), у менеджер "Вика" (12)

## Возможные причины:
- Результаты расчитывались в ручную - проверить метод подсчета
- При выгрузке данных фильтры были неверно настроены или данные округлялись - проверить метод выгрузки
- Была скидка о которой неупомянуто в таблице - добавить скидку, если она есть
- Значения "Сумме с НДС" посчитаны с учетом негодной к продаже продукции - добавить вес не пригодный к продаже

## Число локализованных расхождений

In [5]:
result[
    (result["Месяц"] == 2) &
    (result["Канал сбыта"] == "1.1 Дистрибьюторы") &
    (result["Менеджер"] == "Вика")
].shape[0]

11

## Число по максимальному расхорждению

In [6]:
result[result["Относительное расхождение"] >= 0.3].shape[0]

5

# 2. Кейс. По ссылке находится Массив данных по отдельной номенклатуре. Проанализируйте его, выявите проблемы, обозначьте выводы.

In [7]:
SPREADSHEET_ID = "1Ha-4e1lK294tKyGSqfFoy6H7VebN0Xz9"

df_sales = get_table(SPREADSHEET_ID, "массив продажи", "./ПАРМЕЗАН ПРОДАЖИ_отпр - массив продажи.csv")
df_remaining = get_table(SPREADSHEET_ID, "массив остатки", "./ПАРМЕЗАН ПРОДАЖИ_отпр - массив остатки.csv")

df_sales["Годен до"] = pd.to_datetime(df_sales["Годен до"], dayfirst=True)
df_remaining["Годен до"] = pd.concat([
    pd.to_datetime(df_remaining["Годен до"][:17]), 
    pd.to_datetime(df_remaining["Годен до"][17:], dayfirst=True)
])

result = pd.concat([
    df_sales.groupby("Годен до").agg({"Вес": "sum"}),
    df_remaining.groupby("Годен до").agg({"Вес": "sum"})
], axis=1)
result.columns = ["Вес продано", "Вес остаток"]
result

,Вес продано,Вес остаток
Годен до,,
2025-07-18,3118.555,NaN
2025-08-17,4569.630,NaN
2025-08-23,3326.300,NaN
2025-09-08,13493.645,NaN
2025-10-27,6830.660,NaN
2025-11-23,25627.102,NaN
2025-12-30,10908.553,NaN
2026-01-01,NaN,85000.0
2026-01-13,7812.800,NaN


## Проблема
- только об одном сроке годности есть данные и по объему продаж и по остатку

## Возможные причины:
- Даные небыли занесены в базу - проверить, все ли чеки были загружены, все ли остатки на складе были записаны в базу
- Неправильный фильтр при выгрузке данных - проверить метод выгрузки
- Из-за переезда данных в новую базу старые данные утратились или не догрузились - проверить содержимое старой базы